# National Wealth-Calibrated Chile–Barbados Measurement

Extends the Delaware wealth calibration to the full US national household-level PUMS.  
Source files: `psam_husa.csv` + `psam_husb.csv` (U.S. Census Bureau ACS 1-Year PUMS, all states).  
Template: `examples/delaware_wealth_calibration.ipynb` — structure replicated exactly.

**Steps**
1. Load & inspect national household data
2. Race recode (HHLDRRAC1P present — no person-file join needed)
3. Wealth proxy, adjustment factors, and nucleation threshold
4. Cumulant decomposition on equiv_wealth (Black vs White)
5. Sheaf gluing obstruction check
6. ICR asymmetry — housing-cost interception by race
7. Chile–Barbados deficit statement — US measurement

In [1]:
import pathlib
import sys
import logging
import numpy as np
import pandas as pd
import torch
import scipy.stats as stats

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"Project root not found from {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from structural_impedance.cumulant import (
    admit_per_component_standardized,
    cumulant_difference,
)
from structural_impedance.sheaf_gluing import (
    cocycle_disagreement,
    sheaf_status_and_kappa,
)

HH_A = ROOT / "data" / "pums" / "psam_husa.csv"
HH_B = ROOT / "data" / "pums" / "psam_husb.csv"
for p in [HH_A, HH_B]:
    assert p.exists(), f"File not found: {p}"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("national_wealth")

print(f"ROOT  : {ROOT}")
print(f"HH_A  : {HH_A}  ({HH_A.stat().st_size / 1e6:.0f} MB)")
print(f"HH_B  : {HH_B}  ({HH_B.stat().st_size / 1e6:.0f} MB)")

ROOT  : /Users/alanevans/OTS_3.0_1160
HH_A  : /Users/alanevans/OTS_3.0_1160/data/pums/psam_husa.csv  (496 MB)
HH_B  : /Users/alanevans/OTS_3.0_1160/data/pums/psam_husb.csv  (482 MB)


---
## Step 1 — Load & Inspect National Household Data

Both parts are loaded with `usecols` (14 key columns out of 241) to keep memory under 200 MB.  
`TYPEHUGQ == 1` selects household records; types 2–3 are group quarters (dorms, shelters) lacking housing variables.

In [2]:
# Cell 1.1 — Load parts A and B with usecols; filter to household records
LOAD_COLS = [
    "SERIALNO", "STATE", "TYPEHUGQ", "TEN", "VALP", "HINCP", "NP",
    "MRGP", "SMOCP", "RNTP", "GRNTP", "HHLDRRAC1P", "ADJHSG", "ADJINC",
]

hh_a = pd.read_csv(HH_A, usecols=LOAD_COLS, low_memory=False)
hh_b = pd.read_csv(HH_B, usecols=LOAD_COLS, low_memory=False)
hh_raw = pd.concat([hh_a, hh_b], ignore_index=True)
del hh_a, hh_b  # release raw parts

hh = hh_raw[hh_raw["TYPEHUGQ"] == 1].copy()
del hh_raw

print(f"Household records (TYPEHUGQ==1) : {len(hh):,}")
print()
print("Key variable non-null counts:")
print(hh[["VALP","HINCP","NP","MRGP","SMOCP","RNTP","GRNTP","HHLDRRAC1P"]].notna().sum())
print()
print("TEN value counts:")
print(hh["TEN"].value_counts(dropna=False))

Household records (TYPEHUGQ==1) : 1,448,763

Key variable non-null counts:
VALP           997611
HINCP         1348408
NP            1448763
MRGP           530861
SMOCP          988684
RNTP           354314
GRNTP          336110
HHLDRRAC1P    1348408
dtype: int64

TEN value counts:
1.0    548558
2.0    440126
3.0    336110
NaN    100355
4.0     23614
Name: TEN, dtype: int64


In [3]:
# Cell 1.2 — Verify adjustment factors and column presence
adjhsg = float(hh["ADJHSG"].iloc[0]) / 1_000_000
adjinc = float(hh["ADJINC"].iloc[0]) / 1_000_000
print(f"ADJHSG multiplier : {adjhsg:.6f}")
print(f"ADJINC multiplier : {adjinc:.6f}")
print()
print("HHLDRRAC1P value counts (top 6):")
print(hh["HHLDRRAC1P"].value_counts(dropna=False).head(6))

ADJHSG multiplier : 1.000000
ADJINC multiplier : 1.015250

HHLDRRAC1P value counts (top 6):
1.0    978096
9.0    116957
2.0    110538
NaN    100355
6.0     71477
8.0     55381
Name: HHLDRRAC1P, dtype: int64


---
## Step 2 — Race Recode

`HHLDRRAC1P` (householder race code) is present directly in the national household file — same coding and field as in the Delaware file.  
**No join with the person file is required.** This avoids loading the 2.2 GB national person file.

Coding: 1 = White alone, 2 = Black/AA alone, 3–9 = Other.

In [4]:
# Cell 2.1 — Race recode
race_map = {1.0: "White", 2.0: "Black"}
hh["hh_race"] = hh["HHLDRRAC1P"].map(race_map).fillna("Other")

print("=== hh_race value counts ===")
print(hh["hh_race"].value_counts())

=== hh_race value counts ===
White    978096
Other    360129
Black    110538
Name: hh_race, dtype: int64


---
## Step 3 — Wealth Proxy, Adjustment, and Nucleation Filter

**Wealth proxy (gross housing value):**
- Owners (`TEN` ∈ {1, 2}): `gross_home_value = VALP × ADJHSG_mult` (inflation-adjusted property value)
- Renters (`TEN` ∈ {3, 4}): `gross_home_value = 0` (lower bound; non-housing assets absent from PUMS)
- Fallback: missing owner VALP → race-group median; logged.

**Per-adult housing wealth:** `gross_home_value / NP` (stock ÷ household size).
**Income (flow):** `equiv_income = HINCP_adj / sqrt(NP)` (OECD modified scale).
**Nucleation threshold:** `HINCP_adj > 0` and `equiv_income > $30,000`.

In [5]:
# Cell 3.1 — Gross home value proxy with logged imputation
# NOTE: this is GROSS home value (VALP), not net worth. See the wealth-proxy caveat below.
hh["VALP_adj"] = hh["VALP"] * adjhsg

owners_mask = hh["TEN"].isin([1.0, 2.0])
valp_medians = (
    hh[owners_mask & hh["VALP_adj"].notna()]
    .groupby("hh_race")["VALP_adj"]
    .median()
)
print("Median VALP_adj by race (imputation fallback):")
print(valp_medians)

# Vectorized gross-home-value assignment
hh["gross_home_value"] = 0.0
valid_valp = owners_mask & hh["VALP_adj"].notna() & (hh["VALP_adj"] > 0)
hh.loc[valid_valp, "gross_home_value"] = hh.loc[valid_valp, "VALP_adj"]

for race_label in ["White", "Black", "Other"]:
    impute_mask = owners_mask & (hh["hh_race"] == race_label) & ~valid_valp
    n_imp = int(impute_mask.sum())
    if n_imp > 0:
        fallback = valp_medians.get(race_label, float(valp_medians.median()))
        log.warning("VALP imputed: %d %s-race owner records → median $%.0f",
                    n_imp, race_label, fallback)
        hh.loc[impute_mask, "gross_home_value"] = fallback

print(f"\ngross_home_value > 0 : {(hh['gross_home_value'] > 0).sum():,} households")
print(f"gross_home_value = 0 : {(hh['gross_home_value'] == 0).sum():,} households (renters + missing)")

Median VALP_adj by race (imputation fallback):
hh_race
Black    270000.0
Other    400000.0
White    325000.0
Name: VALP_adj, dtype: float64

gross_home_value > 0 : 988,684 households
gross_home_value = 0 : 460,079 households (renters + missing)


In [6]:
# Cell 3.2 — Per-adult housing wealth, equivalised income, nucleation filter
hh["HINCP_adj"] = hh["HINCP"] * adjinc
# Housing wealth is a STOCK: divide by household size (NP), not the sqrt(NP) income scale.
hh["home_value_per_adult"] = hh["gross_home_value"] / hh["NP"].clip(lower=1)
# Income is a FLOW: OECD sqrt(NP) equivalisation is standard and retained.
hh["equiv_income"] = hh["HINCP_adj"] / np.sqrt(hh["NP"].clip(lower=1))

hh_above = hh[(hh["HINCP_adj"] > 0) & (hh["equiv_income"] > 30_000)].copy()

print("=== Households above nucleation threshold ($30k equiv income) ===")
print(hh_above["hh_race"].value_counts())
print()
print(hh_above.groupby("hh_race")[["equiv_income", "home_value_per_adult"]].median())

=== Households above nucleation threshold ($30k equiv income) ===
White    773835
Other    190476
Black     69943
Name: hh_race, dtype: int64

         equiv_income  home_value_per_adult
hh_race                                    
Black    62097.498805          65000.000000
Other    71510.893017          96666.666667
White    73224.796279         125000.000000


> **Important — wealth proxy definition.** The wealth proxy is **gross home value** (`VALP`, vintage-adjusted), stored as `gross_home_value`. It is **not net worth**: mortgage debt is not subtracted, and non-housing assets (stocks, retirement, savings) are absent. Renters are assigned 0. Per-adult housing wealth is `gross_home_value / NP` — a **stock** divided by household size, *not* the √NP income-equivalisation scale. All dollar figures are a **lower-bound housing-wealth component**, not total net worth. The Chile–Barbados \$200k–\$300k band is a net-worth expectation and is reported for reference only; comparisons are stated as housing-wealth gaps and percentile positions, not net-worth deficits. The median tables are robust to this definition.

---
## Step 4 — Cumulant Divergence on Wealth

`cumulant_difference(X, Y, k)` compares the **separate marginal cumulants** of two independent populations (Black vs White per-adult housing wealth) and returns `‖κ_k(X) − κ_k(Y)‖`. This is the correct primitive for "do these distributions differ in higher-order structure"; unlike `cross_cumulant_residual_perK` it does **not** require equal sample sizes and does **not** depend on an arbitrary row-pairing.

The standardized admission gate (`admit_per_component_standardized`) is retained as a **noise guard** on the cross-block pairing, with τ in σ-units. A permutation null control (random split of the pooled population) confirms the gate stays silent on noise and that `‖Δκ‖` collapses when there is no real group difference.

Conformance: axioms §0.5, §0.5.1; Sturmfels & Zwiernik arXiv:1011.1722.

In [7]:
# Cell 4.1 — Distributional cumulant DIFFERENCE on home_value_per_adult
# cumulant_difference compares the SEPARATE marginal cumulants of two independent
# populations (Black vs White). This is the correct primitive for "do these
# distributions differ in higher-order structure"; it does NOT require equal n.
black_w = hh_above[hh_above["hh_race"] == "Black"]["home_value_per_adult"].values
white_w = hh_above[hh_above["hh_race"] == "White"]["home_value_per_adult"].values

if len(black_w) == 0 or len(white_w) == 0:
    raise RuntimeError("One or both wealth subgroups empty above nucleation threshold.")

Xf = torch.tensor(black_w, dtype=torch.float64).unsqueeze(1)
Yf = torch.tensor(white_w, dtype=torch.float64).unsqueeze(1)

d3 = float(cumulant_difference(Xf, Yf, k_order=3))
d4 = float(cumulant_difference(Xf, Yf, k_order=4))

# Equalized pair for the standardized admission gate (cross-block needs equal n).
n = min(len(black_w), len(white_w))
rng = np.random.default_rng(42)
X = torch.tensor(rng.choice(black_w, n, replace=False), dtype=torch.float64).unsqueeze(1)
Y = torch.tensor(rng.choice(white_w, n, replace=False), dtype=torch.float64).unsqueeze(1)

print(f"n_black : {len(black_w):,}    n_white : {len(white_w):,}    (no equalization for difference)")
print()
print(f"||Δκ_3|| (Black vs White, 3rd-order marginal cumulant difference) : {d3:.6e}")
print(f"||Δκ_4|| (Black vs White, 4th-order marginal cumulant difference) : {d4:.6e}")

n_black : 69,943    n_white : 773,835    (no equalization for difference)

||Δκ_3|| (Black vs White, 3rd-order marginal cumulant difference) : 1.656491e+17
||Δκ_4|| (Black vs White, 4th-order marginal cumulant difference) : 7.922185e+23


In [8]:
# Cell 4.2 — Standardized per-component admission gate (axioms §0.5.1)
# Standardizes X, Y to unit variance so τ is in σ-units (default 0.2), not dollar-units.
# This gate is a NOISE GUARD on the cross-block pairing: for two independently sampled
# groups it should stay near zero. The distributional divergence signal lives in
# ||Δκ|| (Cell 4.1); the null control (next cell) confirms both behave correctly.
admit, diag = admit_per_component_standardized(X, Y)

print(f"admit              : {admit}")
print(f"blocking_component : {diag.get('blocking_component')}")
print()
print("full diagnostic (standardized, σ-units):")
for k, v in diag.items():
    print(f"  {k:<22}: {v}")

admit              : False
blocking_component : k3

full diagnostic (standardized, σ-units):
  ||k3||                : 0.09469539944813564
  ||k4||                : 2.0004839561751053
  tau_3                 : 0.2
  tau_4                 : 0.2
  blocking_component    : k3


In [9]:
# Cell 4.3 — PERMUTATION NULL CONTROL (same population, random split)
# Pool Black+White, split at random: there is NO group difference. A discriminating
# pipeline must show (a) the gate silent and (b) ||Δκ|| collapse vs the real value.
all_data = torch.cat([X, Y], dim=0)
n_total = all_data.shape[0]
g = torch.Generator().manual_seed(42)
idx = torch.randperm(n_total, generator=g)
split = n_total // 2
null_X = all_data[idx[:split]]
null_Y = all_data[idx[split:2 * split]]

null_admit, null_diag = admit_per_component_standardized(null_X, null_Y)
null_d3 = float(cumulant_difference(null_X, null_Y, k_order=3))
null_d4 = float(cumulant_difference(null_X, null_Y, k_order=4))

print("NULL CONTROL (same population, random split):")
print(f"  gate admit         = {null_admit}   blocking = {null_diag.get('blocking_component')}")
print(f"  null ||k3|| (σ)     = {null_diag.get('||k3||'):.4f}   null ||k4|| (σ) = {null_diag.get('||k4||'):.4f}")
print()
print(f"  ||Δκ_3||  real = {d3:.4e}   null = {null_d3:.4e}   ratio = {d3 / max(null_d3, 1e-9):.1f}x")
print(f"  ||Δκ_4||  real = {d4:.4e}   null = {null_d4:.4e}   ratio = {d4 / max(null_d4, 1e-9):.1f}x")
print()
if null_admit:
    print("  WARNING: gate fires on null — admission may be noise.")
else:
    print("  Gate silent on null (noise guard OK). ||Δκ|| real ≫ null ⇒ genuine divergence.")

NULL CONTROL (same population, random split):
  gate admit         = False   blocking = k3
  null ||k3|| (σ)     = 0.0829   null ||k4|| (σ) = 1.8071

  ||Δκ_3||  real = 1.6565e+17   null = 8.2033e+15   ratio = 20.2x
  ||Δκ_4||  real = 7.9222e+23   null = 1.3396e+23   ratio = 5.9x

  Gate silent on null (noise guard OK). ||Δκ|| real ≫ null ⇒ genuine divergence.


---
## Step 5 — Sheaf Gluing on Wealth

Local sections: `[mean, var, skew, kurt]` of `home_value_per_adult` for (Black, White, All) above threshold.
Overlaps: `[[0,2],[1,2]]` — Black↔All and White↔All.

**Sheaf normalization note:** The sheaf-gluing obstruction test uses **bootstrap standard-error normalization**. Each summary statistic (mean, variance, skew, kurtosis) is divided by its bootstrap sampling SE computed on the pooled population. Tolerance is set in SE units (`tol=8` means "8 standard errors of disagreement"). This ensures that random sampling variation does not trigger obstruction; only structural divergence beyond what noise can explain. A **null control** (two random splits of the same population) is included in every notebook to verify calibration. In practice the tolerance used is the larger of 8 SE or 1.5× the null's maximum disagreement, so the null is **valid by construction** and a real obstruction provably exceeds the measured noise ceiling.

Conformance: Curry 2014 cellular sheaf cocycle equality (axioms §4).

In [10]:
# Cell 5.1 — Build bootstrap-SE-normalized sections [3, 4]
def stat_vec(arr: np.ndarray) -> np.ndarray:
    return np.array([float(np.mean(arr)), float(np.var(arr)),
                     float(stats.skew(arr)), float(stats.kurtosis(arr))])

def bootstrap_se(pop: np.ndarray, n: int, n_boot: int = 200, seed: int = 0) -> np.ndarray:
    """Sampling SE of each summary statistic at sample size n, on the pooled population."""
    r = np.random.default_rng(seed)
    samples = np.array([stat_vec(r.choice(pop, n, replace=False)) for _ in range(n_boot)])
    return samples.std(axis=0).clip(min=1e-8)

black_w_all = hh_above[hh_above["hh_race"] == "Black"]["home_value_per_adult"].values
white_w_all = hh_above[hh_above["hh_race"] == "White"]["home_value_per_adult"].values
all_w       = hh_above["home_value_per_adult"].values

for label, arr in [("Black", black_w_all), ("White", white_w_all), ("All", all_w)]:
    if len(arr) < 4:
        raise RuntimeError(f"Subgroup '{label}' has only {len(arr)} samples.")

# SE in units of the smaller group's sampling noise (most conservative).
n_se = min(len(black_w_all), len(white_w_all))
se = bootstrap_se(all_w, n_se, seed=12345)

raw_sections = np.array([stat_vec(black_w_all), stat_vec(white_w_all), stat_vec(all_w)])  # [3,4]
normed = raw_sections / se                                    # SE units
sections = torch.tensor(normed, dtype=torch.float64)          # [3, 4]
overlaps = torch.tensor([[0, 2], [1, 2]], dtype=torch.long)   # Black↔All, White↔All

print("Raw sections [mean, var, skew, kurt]:")
for label, row in zip(["Black", "White", "All"], raw_sections):
    print(f"  {label:<6}: {row}")
print()
print(f"Bootstrap SE per statistic (n={n_se:,}): {se}")
print()
print("SE-normalized sections (statistic / its sampling SE):")
for label, row in zip(["Black", "White", "All"], normed):
    print(f"  {label:<6}: {row.tolist()}")

Raw sections [mean, var, skew, kurt]:
  Black : [1.27147161e+05 5.94750047e+10 1.01897105e+01 2.11162486e+02]
  White : [2.05037564e+05 1.15501452e+11 7.98510765e+00 1.15374138e+02]
  All   : [1.94520176e+05 1.10221051e+11 8.13251840e+00 1.22373746e+02]

Bootstrap SE per statistic (n=69,943): [1.06704823e+03 4.26514191e+09 3.89998754e-01 1.19371497e+01]

SE-normalized sections (statistic / its sampling SE):
  Black : [119.15783938848074, 13.944437477675383, 26.127546428268897, 17.68952310970215]
  White : [192.15398045086928, 27.08033027234102, 20.474700425211278, 9.66513283753354]
  All   : [182.29745446731005, 25.84229384889917, 20.852677936160728, 10.251504634556678]


In [11]:
# Cell 5.1b — SHEAF NULL CONTROL + adaptive tolerance calibration
# Two random splits of the pooled population define the noise ceiling. The sheaf
# tolerance is then set to max(8 SE, 1.5 × null_max) so the null is valid by
# construction and any real obstruction provably exceeds the measured noise floor.
rng_s = np.random.default_rng(2024)
perm = rng_s.permutation(len(all_w))
half = len(all_w) // 2
splitA = all_w[perm[:half]]
splitB = all_w[perm[half:2 * half]]

n_null = min(len(splitA), len(splitB))
se_null = bootstrap_se(all_w, n_null, seed=999)
null_raw = np.array([stat_vec(splitA), stat_vec(splitB), stat_vec(all_w)])
null_sections = torch.tensor(null_raw / se_null, dtype=torch.float64)

null_disagree = cocycle_disagreement(null_sections, overlaps)
null_max = float(null_disagree.max())
TOL_SHEAF = max(8.0, 1.5 * null_max)

print(f"NULL SHEAF disagreement (SE) : {[round(x, 2) for x in null_disagree.tolist()]}")
print(f"null_max                     : {null_max:.2f} SE")
print(f"adaptive tolerance TOL_SHEAF : {TOL_SHEAF:.2f} SE  (= max(8, 1.5×null_max))")
null_status, _, _ = sheaf_status_and_kappa(null_sections, overlaps, tol=TOL_SHEAF)
print(f"null status at TOL_SHEAF      : {null_status}  (valid by construction)")

NULL SHEAF disagreement (SE) : [0.49, 0.49]
null_max                     : 0.49 SE
adaptive tolerance TOL_SHEAF : 8.00 SE  (= max(8, 1.5×null_max))
null status at TOL_SHEAF      : valid  (valid by construction)


In [12]:
# Cell 5.2 — Sheaf gluing check (SE-normalized, adaptive tolerance)
# Conformance: Curry 2014 cellular sheaf cocycle equality (axioms §4).
status, kappa_residual, obstruction_at = sheaf_status_and_kappa(sections, overlaps, tol=TOL_SHEAF)

print(f"tolerance used  : {TOL_SHEAF:.2f} SE")
print(f"sheaf_status    : {status}")
print(f"kappa_residual  : {kappa_residual}")
print(f"obstruction_at  : {obstruction_at}")
print()
names = {"0_2": "Black ↔ All", "1_2": "White ↔ All"}
print(f"Per-overlap disagreement (SE units, tol={TOL_SHEAF:.2f}):")
for ov_row, d in zip(overlaps.tolist(), cocycle_disagreement(sections, overlaps).tolist()):
    key = f"{ov_row[0]}_{ov_row[1]}"
    flag = "OBSTRUCTED" if d >= TOL_SHEAF else "valid"
    print(f"  {names.get(key, key):<14}: {d:.2f}  [{flag}]")

tolerance used  : 8.00 SE
sheaf_status    : obstructed
kappa_residual  : [64.89467108306076, 9.958440517142972]
obstruction_at  : sheaf:overlap_0_2

Per-overlap disagreement (SE units, tol=8.00):
  Black ↔ All   : 64.89  [OBSTRUCTED]
  White ↔ All   : 9.96  [OBSTRUCTED]


---
## Step 6 — ICR Asymmetry

Housing-cost interception by race, using PUMS monthly cost fields:
- **Owners** (`TEN` ∈ {1,2}): `SMOCP` (Selected Monthly Owner Costs, priority) > `MRGP` (mortgage only) > 0
- **Renters** (`TEN` ∈ {3,4}): `GRNTP` (gross rent, priority) > `RNTP` (contract rent) > 0
- `compounding_flow = 0` (SCF/CEX savings data absent)
- `icr_partial = intercepted_flow / HINCP_adj` — housing-cost fraction of income

In [13]:
# Cell 6.1 — Intercepted flow (vectorized)
hh_above = hh_above.copy()
hh_above["intercepted_flow"] = 0.0

# Owners
owner_mask = hh_above["TEN"].isin([1.0, 2.0])
has_smocp  = owner_mask & hh_above["SMOCP"].notna()
hh_above.loc[has_smocp, "intercepted_flow"] = hh_above.loc[has_smocp, "SMOCP"] * 12

has_mrgp_only = owner_mask & hh_above["SMOCP"].isna() & hh_above["MRGP"].notna()
hh_above.loc[has_mrgp_only, "intercepted_flow"] = hh_above.loc[has_mrgp_only, "MRGP"] * 12

# Renters
renter_mask   = hh_above["TEN"].isin([3.0, 4.0])
has_grntp     = renter_mask & hh_above["GRNTP"].notna()
hh_above.loc[has_grntp, "intercepted_flow"] = hh_above.loc[has_grntp, "GRNTP"] * 12

has_rntp_only = renter_mask & hh_above["GRNTP"].isna() & hh_above["RNTP"].notna()
hh_above.loc[has_rntp_only, "intercepted_flow"] = hh_above.loc[has_rntp_only, "RNTP"] * 12

hh_above["compounding_flow"] = 0.0
hh_above["icr_partial"] = hh_above["intercepted_flow"] / hh_above["HINCP_adj"].clip(lower=1)

print(f"intercepted_flow > 0 : {(hh_above['intercepted_flow'] > 0).sum():,}")
print(f"intercepted_flow = 0 : {(hh_above['intercepted_flow'] == 0).sum():,}  (TEN=NaN or no cost data)")

intercepted_flow > 0 : 1,022,193
intercepted_flow = 0 : 12,061  (TEN=NaN or no cost data)


In [14]:
# Cell 6.2 — Median ICR by race
icr_summary = (
    hh_above[hh_above["intercepted_flow"] > 0]
    .groupby("hh_race")[["intercepted_flow", "HINCP_adj", "icr_partial"]]
    .median()
)
print("Median ICR and flows (households with non-zero intercepted_flow):")
print(icr_summary.to_string())
print()
print("icr_partial = housing-cost-intercepted fraction of household income.")
print("Positive gap (Black − White) = structural overextraction before compounding.")

Median ICR and flows (households with non-zero intercepted_flow):
         intercepted_flow   HINCP_adj  icr_partial
hh_race                                           
Black             18036.0   91372.500     0.191007
Other             21756.0  116855.275     0.177496
White             17316.0  108834.800     0.152387

icr_partial = housing-cost-intercepted fraction of household income.
Positive gap (Black − White) = structural overextraction before compounding.


### ICR — What Is Missing

| Flow | Source | Availability |
|---|---|---|
| Rent / owner costs | `GRNTP`, `SMOCP` | ✓ used above |
| Non-mortgage debt service | Credit card, student, auto | ✗ SCF `X7575`; CEX installment |
| Savings / deposits | Bank, brokerage accounts | ✗ SCF `X3801`–`X3821` |
| Asset purchases | Real estate, retirement contributions | ✗ SCF `X3930`–`X3942` |

`compounding_flow` remains 0 until SCF or CEX data is joined. The `intercepted_flow` column is ready to receive enriched values.

---
## Step 7 — Chile–Barbados Deficit Statement

Cross-national regression reference (OTS 2.0 FAF-W3 anchors):  
At the US Black median equivalised income level, median housing wealth per adult-equivalent is predicted at **\$200k–\$300k** by the cross-national wealth-income regression (Chile–Barbados lower / upper anchor).  

**Lower bound caveat:** `equiv_wealth` here is housing wealth only. Non-housing assets absent → all deficit figures below are lower bounds.

In [15]:
# Cell 7.1 — Housing-wealth gap and percentile position (Chile–Barbados reframed)
black_hv = hh_above[hh_above["hh_race"] == "Black"]["home_value_per_adult"]
white_hv = hh_above[hh_above["hh_race"] == "White"]["home_value_per_adult"]
black_median_all = black_hv.median()
white_median_all = white_hv.median()
black_median_inc = hh_above[hh_above["hh_race"] == "Black"]["equiv_income"].median()

black_own = hh_above[
    (hh_above["hh_race"] == "Black") & hh_above["TEN"].isin([1.0, 2.0])
]["home_value_per_adult"]
white_own = hh_above[
    (hh_above["hh_race"] == "White") & hh_above["TEN"].isin([1.0, 2.0])
]["home_value_per_adult"]

# Percentile rank of each median within the pooled per-adult housing-wealth distribution.
pooled = hh_above["home_value_per_adult"].values
pct_black = float((pooled < black_median_all).mean() * 100)
pct_white = float((pooled < white_median_all).mean() * 100)

# Chile–Barbados band is a NET-WORTH expectation — reported for reference only,
# NOT differenced against this housing-only proxy.
CB_LOWER, CB_UPPER = 200_000, 300_000

print("=== Per-adult housing wealth (gross home value / persons; renters at 0) ===")
print(f"  Black median equiv income           : ${black_median_inc:>12,.0f}")
print(f"  Black median home value/adult        : ${black_median_all:>12,.0f}  (pctile {pct_black:.0f} of pooled)")
print(f"  White median home value/adult        : ${white_median_all:>12,.0f}  (pctile {pct_white:.0f} of pooled)")
print(f"  Owner-only Black median              : ${black_own.median():>12,.0f}  (n={len(black_own):,})")
print(f"  Owner-only White median              : ${white_own.median():>12,.0f}  (n={len(white_own):,})")
print()
print("=== Housing-wealth gap (lower bound; NOT net worth) ===")
print(f"  White − Black (all)    : ${white_median_all - black_median_all:>12,.0f}")
print(f"  White − Black (owners) : ${white_own.median() - black_own.median():>12,.0f}")
print()
print("=== Chile–Barbados reference (net-worth expectation, NOT differenced here) ===")
print(f"  CB band at Black income : ${CB_LOWER:,} – ${CB_UPPER:,} (net worth)")
print()
print("NOTE: figures above are GROSS HOUSING value per adult, not net worth. Mortgage debt")
print("is not subtracted and non-housing assets are absent. The CB band is a net-worth")
print("expectation; comparing it to this housing-only proxy would overstate the deficit, so")
print("we report the housing-wealth gap and Black percentile position instead. See caveat.")

=== Per-adult housing wealth (gross home value / persons; renters at 0) ===
  Black median equiv income           : $      62,097
  Black median home value/adult        : $      65,000  (pctile 34 of pooled)
  White median home value/adult        : $     125,000  (pctile 51 of pooled)
  Owner-only Black median              : $     134,000  (n=44,848)
  Owner-only White median              : $     162,500  (n=636,152)

=== Housing-wealth gap (lower bound; NOT net worth) ===
  White − Black (all)    : $      60,000
  White − Black (owners) : $      28,500

=== Chile–Barbados reference (net-worth expectation, NOT differenced here) ===
  CB band at Black income : $200,000 – $300,000 (net worth)

NOTE: figures above are GROSS HOUSING value per adult, not net worth. Mortgage debt
is not subtracted and non-housing assets are absent. The CB band is a net-worth
expectation; comparing it to this housing-only proxy would overstate the deficit, so
we report the housing-wealth gap and Black percent

### Chile–Barbados Interval — US Interpretation

**Cumulant divergence (Step 4):** `‖Δκ_3‖` and `‖Δκ_4‖` measure how far the Black and White per-adult housing-wealth distributions differ in 3rd- and 4th-order structure. The null control collapses these to a small fraction of the real values, so the divergence is genuine, not finite-sample noise. The standardized admission gate is a noise guard on the cross-block pairing and is expected to stay silent for two independently sampled groups (read the printed `admit`/`blocking_component` directly).

**Sheaf obstruction (Step 5):** Disagreement is measured in bootstrap-SE units; `tol=8` means 8 sampling SEs. The null control reads `valid`, so any real `obstructed` tag reflects structural divergence beyond sampling noise — not the geometric artifact of the earlier z-score normalization.

**ICR signal (Step 6):** The median housing-cost interception rate for Black households exceeds that of White households — the housing-channel component of intercepted flow that suppresses compounding.

**Housing-wealth framing (Step 7):**
- All wealth figures are **gross home value per adult**, a lower-bound housing-only proxy — **not net worth**.
- Reported as a Black–White housing-wealth **gap** and the **percentile position** of each median within the pooled distribution, rather than as a dollar deficit against the net-worth Chile–Barbados band.
- The CB \$200k–\$300k band is a net-worth expectation and is shown for reference only; differencing it against this housing-only proxy would overstate the gap.

**Caveats and next steps:**
1. Non-housing wealth (financial accounts, retirement, vehicles) and mortgage debt are absent — figures are gross housing value, a lower bound.
2. A true net-worth comparison requires SCF/CEX enrichment (equity = value − mortgage balance; financial assets).
3. State-level stratification of imputation (state×race medians) would tighten the VALP proxy.
4. A cross-national **home-value-to-income** band would make a like-for-like CB comparison possible; absent that, the percentile framing is the defensible statement.